In [5]:
!pip install mediapipe tensorflow scikit-learn opencv-python numpy pandas -q

In [9]:
import os
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
import urllib.request
from pathlib import Path

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


In [10]:
MODELO_PATH = 'hand_landmarker.task'

if not os.path.exists(MODELO_PATH):
    print('⏳ Descargando modelo MediaPipe...')
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task',
        MODELO_PATH
    )
    print('✅ Modelo descargado')
else:
    print('✅ Modelo ya existe')

⏳ Descargando modelo MediaPipe...
✅ Modelo descargado


In [11]:
from mediapipe.tasks.python.vision import HandLandmarker, HandLandmarkerOptions, RunningMode

opciones = HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=MODELO_PATH),
    running_mode=RunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.5
)
detector = HandLandmarker.create_from_options(opciones)

print("✅ Detector de manos listo")

✅ Detector de manos listo


In [12]:
import urllib.request

url = "https://zenodo.org/records/10067509/files/MSL-ABC.7z?download=1"
destino = "MSL-ABC.7z"

if not os.path.exists(destino):
    print("⏳ Descargando dataset... (puede tardar varios minutos)")
    urllib.request.urlretrieve(url, destino)
    print("✅ Dataset descargado")
else:
    print("✅ Dataset ya existe")

⏳ Descargando dataset... (puede tardar varios minutos)
✅ Dataset descargado


In [13]:
!pip install py7zr -q
print("✅ py7zr instalado")

✅ py7zr instalado


In [17]:
import py7zr

if not os.path.exists('lsm_dataset'):
    print("⏳ Extrayendo dataset... (puede tardar unos minutos)")
    try:
        with py7zr.SevenZipFile('MSL-ABC.7z', mode='r') as z:
            z.extractall(path='lsm_dataset')
        print("✅ Dataset extraído")
    except Exception as e:
        print(f"❌ Error al extraer: {e}")
else:
    print("✅ Dataset ya existe")

✅ Dataset ya existe


In [18]:
for raiz, carpetas, archivos in os.walk('lsm_dataset'):
    nivel = raiz.replace('lsm_dataset', '').count(os.sep)
    if nivel < 3:
        indent = ' ' * 2 * nivel
        print(f'{indent}{os.path.basename(raiz)}/')
        if archivos:
            print(f'{indent}  → {len(archivos)} archivos')

lsm_dataset/
  MSL-ABC/
    lsm-abc-A/
    lsm-abc-B/
    lsm-abc-C/


In [19]:
for raiz, carpetas, archivos in os.walk('lsm_dataset/MSL-ABC/lsm-abc-A'):
    nivel = raiz.replace('lsm_dataset/MSL-ABC/lsm-abc-A', '').count(os.sep)
    if nivel < 3:
        indent = ' ' * 2 * nivel
        print(f'{indent}{os.path.basename(raiz)}/')
        if archivos:
            print(f'{indent}  → {len(archivos)} archivos (ej: {archivos[0]})')

lsm-abc-A/
  test/
    A/
      → 435 archivos (ej: S18-A-4-0.jpg)
    B/
      → 441 archivos (ej: S18-B-4-0.jpg)
    C/
      → 438 archivos (ej: S18-C-4-0.jpg)
    D/
      → 453 archivos (ej: S18-D-4-0.jpg)
    E/
      → 471 archivos (ej: S18-E-4-0.jpg)
    F/
      → 456 archivos (ej: S18-F-4-0.jpg)
    G/
      → 441 archivos (ej: S18-G-4-0.jpg)
    H/
      → 441 archivos (ej: S18-H-4-0.jpg)
    I/
      → 447 archivos (ej: S18-I-4-0.jpg)
    L/
      → 486 archivos (ej: S18-L-4-0.jpg)
    M/
      → 455 archivos (ej: S18-M-4-0.jpg)
    N/
      → 455 archivos (ej: S18-N-4-1.jpg)
    O/
      → 459 archivos (ej: S18-O-4-0.jpg)
    P/
      → 444 archivos (ej: S18-P-4-0.jpg)
    R/
      → 438 archivos (ej: S18-R-4-0.jpg)
    S/
      → 435 archivos (ej: S18-S-4-0.jpg)
    T/
      → 438 archivos (ej: S18-T-4-0.jpg)
    U/
      → 444 archivos (ej: S18-U-4-0.jpg)
    V/
      → 450 archivos (ej: S18-V-4-0.jpg)
    W/
      → 441 archivos (ej: S18-W-4-0.jpg)
    Y/
      → 471 ar

In [2]:
def extraer_keypoints(ruta_imagen):
    imagen = cv2.imread(str(ruta_imagen))
    if imagen is None:
        return None
    imagen_rgb = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
    mp_imagen = mp.Image(image_format=mp.ImageFormat.SRGB, data=imagen_rgb)
    resultado = detector.detect(mp_imagen)
    if resultado.hand_landmarks:
        keypoints = []
        for punto in resultado.hand_landmarks[0]:
            keypoints.extend([punto.x, punto.y, punto.z])
        return keypoints
    return None

print("✅ Función definida")

✅ Función definida


In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
import urllib.request
from pathlib import Path
from mediapipe.tasks.python.vision import HandLandmarker, HandLandmarkerOptions, RunningMode

MODELO_PATH = 'hand_landmarker.task'

opciones = HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=MODELO_PATH),
    running_mode=RunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.5
)
detector = HandLandmarker.create_from_options(opciones)

def extraer_keypoints(ruta_imagen):
    imagen = cv2.imread(str(ruta_imagen))
    if imagen is None:
        return None
    imagen_rgb = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
    mp_imagen = mp.Image(image_format=mp.ImageFormat.SRGB, data=imagen_rgb)
    resultado = detector.detect(mp_imagen)
    if resultado.hand_landmarks:
        keypoints = []
        for punto in resultado.hand_landmarks[0]:
            keypoints.extend([punto.x, punto.y, punto.z])
        return keypoints
    return None

print("✅ Todo listo")

✅ Todo listo


In [6]:
datos = []
imagenes_ok = 0
imagenes_fail = 0

rutas = list(Path('lsm_dataset/MSL-ABC').rglob('*.jpg'))
total = len(rutas)
print(f'Total de imágenes encontradas: {total}')

for i, ruta_imagen in enumerate(rutas):
    if i % 5000 == 0:
        print(f'Progreso: {i}/{total} ({i/total*100:.1f}%)')

    etiqueta = ruta_imagen.parent.name
    if len(etiqueta) != 1 or not etiqueta.isalpha():
        continue

    keypoints = extraer_keypoints(ruta_imagen)
    if keypoints is not None:
        datos.append(keypoints + [etiqueta])
        imagenes_ok += 1
    else:
        imagenes_fail += 1

print(f'\n✅ Detectadas: {imagenes_ok}')
print(f'❌ Sin detección: {imagenes_fail}')
print(f'📊 Éxito: {imagenes_ok/(imagenes_ok+imagenes_fail)*100:.1f}%')

Total de imágenes encontradas: 279716
Progreso: 0/279716 (0.0%)
Progreso: 5000/279716 (1.8%)
Progreso: 10000/279716 (3.6%)
Progreso: 15000/279716 (5.4%)
Progreso: 20000/279716 (7.2%)
Progreso: 25000/279716 (8.9%)
Progreso: 30000/279716 (10.7%)
Progreso: 35000/279716 (12.5%)
Progreso: 40000/279716 (14.3%)
Progreso: 45000/279716 (16.1%)
Progreso: 50000/279716 (17.9%)
Progreso: 55000/279716 (19.7%)
Progreso: 60000/279716 (21.5%)
Progreso: 65000/279716 (23.2%)
Progreso: 70000/279716 (25.0%)
Progreso: 75000/279716 (26.8%)
Progreso: 80000/279716 (28.6%)
Progreso: 85000/279716 (30.4%)
Progreso: 90000/279716 (32.2%)
Progreso: 95000/279716 (34.0%)
Progreso: 100000/279716 (35.8%)
Progreso: 105000/279716 (37.5%)
Progreso: 110000/279716 (39.3%)
Progreso: 115000/279716 (41.1%)
Progreso: 120000/279716 (42.9%)
Progreso: 125000/279716 (44.7%)
Progreso: 130000/279716 (46.5%)
Progreso: 135000/279716 (48.3%)
Progreso: 140000/279716 (50.1%)
Progreso: 145000/279716 (51.8%)
Progreso: 150000/279716 (53.6%)
P

In [ ]:
CELDA A CORRER CADA QUE EMPIEZE A COMPILAR EL MODELO DE NUEZ

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
import urllib.request
from pathlib import Path
from mediapipe.tasks.python.vision import HandLandmarker, HandLandmarkerOptions, RunningMode

# Configurar detector de manos
MODELO_PATH = 'hand_landmarker.task'
opciones = HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=MODELO_PATH),
    running_mode=RunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.5
)
detector = HandLandmarker.create_from_options(opciones)

# Función para extraer keypoints
def extraer_keypoints(ruta_imagen):
    imagen = cv2.imread(str(ruta_imagen))
    if imagen is None:
        return None
    imagen_rgb = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
    mp_imagen = mp.Image(image_format=mp.ImageFormat.SRGB, data=imagen_rgb)
    resultado = detector.detect(mp_imagen)
    if resultado.hand_landmarks:
        keypoints = []
        for punto in resultado.hand_landmarks[0]:
            keypoints.extend([punto.x, punto.y, punto.z])
        return keypoints
    return None

# Cargar CSV si ya existe
if os.path.exists('keypoints.csv'):
    df = pd.read_csv('keypoints.csv')
    print(f'✅ CSV cargado: {len(df)} filas')
else:
    df = None
    print('⚠️ No hay CSV todavía, necesitas procesar las imágenes primero')

print("✅ Todo listo para trabajar")

✅ CSV cargado: 276113 filas
✅ Todo listo para trabajar


In [5]:
import csv

datos = []
imagenes_ok = 0
imagenes_fail = 0

columnas = [f'{eje}{i}' for i in range(21) for eje in ['x', 'y', 'z']] + ['letra']

# Crear CSV desde el inicio
with open('keypoints.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(columnas)

rutas = list(Path('lsm_dataset/MSL-ABC').rglob('*.jpg'))
total = len(rutas)
print(f'Total de imágenes: {total}')

for i, ruta_imagen in enumerate(rutas):
    if i % 5000 == 0:
        print(f'Progreso: {i}/{total} ({i/total*100:.1f}%)')

    etiqueta = ruta_imagen.parent.name
    if len(etiqueta) != 1 or not etiqueta.isalpha():
        continue

    keypoints = extraer_keypoints(ruta_imagen)
    if keypoints is not None:
        datos.append(keypoints + [etiqueta])
        imagenes_ok += 1
    else:
        imagenes_fail += 1

    # Guardar cada 10,000 imágenes
    if imagenes_ok % 10000 == 0 and imagenes_ok > 0:
        with open('keypoints.csv', 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerows(datos)
        datos = []
        print(f'💾 Guardado parcial: {imagenes_ok} imágenes')

# Guardar el resto
if datos:
    with open('keypoints.csv', 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerows(datos)

print(f'\n✅ Detectadas: {imagenes_ok}')
print(f'❌ Sin detección: {imagenes_fail}')
print(f'📊 Éxito: {imagenes_ok/(imagenes_ok+imagenes_fail)*100:.1f}%')
print(f'💾 CSV guardado correctamente')

Total de imágenes: 279716
Progreso: 0/279716 (0.0%)
Progreso: 5000/279716 (1.8%)
Progreso: 10000/279716 (3.6%)
💾 Guardado parcial: 10000 imágenes
Progreso: 15000/279716 (5.4%)
Progreso: 20000/279716 (7.2%)
💾 Guardado parcial: 20000 imágenes
Progreso: 25000/279716 (8.9%)
Progreso: 30000/279716 (10.7%)
💾 Guardado parcial: 30000 imágenes
Progreso: 35000/279716 (12.5%)
Progreso: 40000/279716 (14.3%)
💾 Guardado parcial: 40000 imágenes
Progreso: 45000/279716 (16.1%)
Progreso: 50000/279716 (17.9%)
💾 Guardado parcial: 50000 imágenes
Progreso: 55000/279716 (19.7%)
Progreso: 60000/279716 (21.5%)
💾 Guardado parcial: 60000 imágenes
Progreso: 65000/279716 (23.2%)
Progreso: 70000/279716 (25.0%)
💾 Guardado parcial: 70000 imágenes
Progreso: 75000/279716 (26.8%)
Progreso: 80000/279716 (28.6%)
💾 Guardado parcial: 80000 imágenes
Progreso: 85000/279716 (30.4%)
Progreso: 90000/279716 (32.2%)
💾 Guardado parcial: 90000 imágenes
Progreso: 95000/279716 (34.0%)
Progreso: 100000/279716 (35.8%)
💾 Guardado parcial

In [2]:
tamaño = os.path.getsize('keypoints.csv') / (1024*1024)
df = pd.read_csv('keypoints.csv')
print(f'📁 CSV pesa: {tamaño:.1f} MB')
print(f'📊 Filas: {len(df)}')
print(f'📋 Columnas: {len(df.columns)}')
print(df['letra'].value_counts().sort_index())

📁 CSV pesa: 328.8 MB
📊 Filas: 276113
📋 Columnas: 64
letra
A    13818
B    13665
C    12916
D    13458
E    13429
F    13565
G    12971
H    12279
I    13479
L    13424
M    13001
N    12203
O    13057
P    13033
R    13296
S    12993
T    13500
U    13062
V    13019
W    13090
Y    12855
Name: count, dtype: int64


In [ ]:
//CELDA INICIAL X2

In [9]:
import os
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
import urllib.request
from pathlib import Path
from mediapipe.tasks.python.vision import HandLandmarker, HandLandmarkerOptions, RunningMode

# Configurar detector de manos
MODELO_PATH = 'hand_landmarker.task'
opciones = HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=MODELO_PATH),
    running_mode=RunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.5
)
detector = HandLandmarker.create_from_options(opciones)

# Función para extraer keypoints
def extraer_keypoints(ruta_imagen):
    imagen = cv2.imread(str(ruta_imagen))
    if imagen is None:
        return None
    imagen_rgb = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
    mp_imagen = mp.Image(image_format=mp.ImageFormat.SRGB, data=imagen_rgb)
    resultado = detector.detect(mp_imagen)
    if resultado.hand_landmarks:
        keypoints = []
        for punto in resultado.hand_landmarks[0]:
            keypoints.extend([punto.x, punto.y, punto.z])
        return keypoints
    return None

# Cargar CSV
if os.path.exists('keypoints.csv'):
    df = pd.read_csv('keypoints.csv')
    print(f'✅ CSV cargado: {len(df)} filas')
else:
    print('⚠️ No hay CSV todavía')

print("✅ Todo listo para trabajar")

FileNotFoundError: Unable to open file at hand_landmarker.task

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Separar features (X) y etiquetas (y)
X = df.drop('letra', axis=1).values
y = df['letra'].values

# Convertir letras a números (A=0, B=1, etc.)
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

# Dividir en train y test
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f'✅ Datos listos')
print(f'   Train: {X_train.shape}')
print(f'   Test:  {X_test.shape}')
print(f'   Clases: {encoder.classes_}')

✅ Datos listos
   Train: (220890, 63)
   Test:  (55223, 63)
   Clases: ['A' 'B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'L' 'M' 'N' 'O' 'P' 'R' 'S' 'T' 'U'
 'V' 'W' 'Y']


In [5]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

modelo = Sequential([
    Dense(256, activation='relu', input_shape=(63,)),
    BatchNormalization(),
    Dropout(0.3),
    
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    
    Dense(64, activation='relu'),
    Dropout(0.2),
    
    Dense(21, activation='softmax')
])

modelo.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

modelo.summary()

C:\Users\c.villalobos-ortiz\.conda\envs\lsm\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 256)                 │          16,384 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 256)                 │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 128)                 │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 21)                  │           1,365 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 60,437 (236.08 KB)

 Trainable params: 59,669 (233.08 KB)

 Non-trainable params: 768 (3.00 KB)

In [6]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint('modelo_lsm.keras', save_best_only=True, verbose=1)
]

historial = modelo.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

print("✅ Entrenamiento terminado")

Epoch 1/50
6211/6213 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7971 - loss: 0.6369
Epoch 1: val_loss improved from None to 0.41562, saving model to modelo_lsm.keras
6213/6213 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - accuracy: 0.8527 - loss: 0.4445 - val_accuracy: 0.8434 - val_loss: 0.4156
Epoch 2/50
6205/6213 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8757 - loss: 0.3674
Epoch 2: val_loss improved from 0.41562 to 0.26265, saving model to modelo_lsm.keras
6213/6213 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - accuracy: 0.8758 - loss: 0.3675 - val_accuracy: 0.9074 - val_loss: 0.2626
Epoch 3/50
6191/6213 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8710 - loss: 0.3802
Epoch 3: val_loss improved from 0.26265 to 0.24112, saving model to modelo_lsm.keras
6213/6213 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - accuracy: 0.8725 - loss: 0.3790 - val_accuracy: 0.9174 - val_loss: 0.2411
Epoch 4/50
6197/6213 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8701 - loss: 0.3869
Epoch 4: val_loss improved from 0.24

In [7]:
from sklearn.metrics import classification_report

# Evaluar en el set de test
loss, accuracy = modelo.evaluate(X_test, y_test, verbose=0)
print(f'✅ Precisión en test: {accuracy*100:.2f}%')
print(f'📉 Loss en test: {loss:.4f}')

# Reporte detallado por letra
y_pred = modelo.predict(X_test, verbose=0)
y_pred_clases = y_pred.argmax(axis=1)

print('\n📊 Reporte por letra:')
print(classification_report(y_test, y_pred_clases, target_names=encoder.classes_))

✅ Precisión en test: 92.68%
📉 Loss en test: 0.2069

📊 Reporte por letra:
              precision    recall  f1-score   support

           A       0.94      0.90      0.92      2764
           B       0.94      0.95      0.94      2733
           C       0.94      1.00      0.97      2583
           D       0.92      0.90      0.91      2692
           E       0.85      0.94      0.89      2686
           F       0.96      0.94      0.95      2713
           G       0.98      0.98      0.98      2594
           H       0.97      0.98      0.97      2456
           I       1.00      0.96      0.98      2696
           L       1.00      0.91      0.95      2685
           M       0.90      0.92      0.91      2600
           N       0.92      0.90      0.91      2440
           O       0.93      0.83      0.88      2611
           P       0.91      0.98      0.94      2607
           R       0.95      0.79      0.86      2659
           S       0.88      0.90      0.89      2599
        

In [7]:
import pickle

with open('encoder_lsm.pkl', 'wb') as f:
    pickle.dump(encoder, f)

print("✅ Encoder guardado")

# Verificar que ambos archivos existen
print(f"📁 modelo_lsm.keras: {os.path.getsize('modelo_lsm.keras')/1024:.1f} KB")
print(f"📁 encoder_lsm.pkl: {os.path.getsize('encoder_lsm.pkl')/1024:.1f} KB")

✅ Encoder guardado


FileNotFoundError: [WinError 2] El sistema no puede encontrar el archivo especificado: 'modelo_lsm.keras'

In [14]:
import tensorflow as tf
import pickle

# Cargar modelo y encoder
modelo = tf.keras.models.load_model('./models/v1/modelo_lsm.keras')
with open('./models/v1/encoder_lsm.pkl', 'rb') as f:
    encoder = pickle.load(f)

cap = cv2.VideoCapture(0)
print("✅ Cámara abierta — presiona Q para salir")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Detectar mano
    imagen_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_imagen = mp.Image(image_format=mp.ImageFormat.SRGB, data=imagen_rgb)
    resultado = detector.detect(mp_imagen)

    letra_predicha = "?"
    confianza = 0.0

    if resultado.hand_landmarks:
        # Extraer keypoints
        keypoints = []
        for punto in resultado.hand_landmarks[0]:
            keypoints.extend([punto.x, punto.y, punto.z])

        # Predecir letra
        X = np.array(keypoints).reshape(1, -1)
        prediccion = modelo.predict(X, verbose=0)
        clase = prediccion.argmax()
        confianza = prediccion[0][clase]
        letra_predicha = encoder.classes_[clase]

        # Dibujar puntos de la mano
        for punto in resultado.hand_landmarks[0]:
            cx = int(punto.x * frame.shape[1])
            cy = int(punto.y * frame.shape[0])
            cv2.circle(frame, (cx, cy), 4, (0, 255, 0), -1)

    # Mostrar letra y confianza
    color = (0, 255, 0) if confianza > 0.8 else (0, 165, 255)
    cv2.putText(frame, f'Letra: {letra_predicha}', (20, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 2, color, 3)
    cv2.putText(frame, f'Confianza: {confianza*100:.1f}%', (20, 120),
                cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

    cv2.imshow('LSM - Reconocimiento', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("✅ Cámara cerrada")

✅ Cámara abierta — presiona Q para salir
✅ Cámara cerrada


In [12]:
import os
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks.python.vision import HandLandmarker, HandLandmarkerOptions, RunningMode
import tensorflow as tf
import pickle

BASE = r'C:\Users\c.villalobos-ortiz\Desktop\AIModel'
os.chdir(BASE)

# Recargar detector
MODELO_PATH = 'hand_landmarker.task'
opciones = HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=MODELO_PATH),
    running_mode=RunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.5
)
detector = HandLandmarker.create_from_options(opciones)
print('✅ Detector listo')

✅ Detector listo
